In [ ]:
# run this notebook after 17_determine_af_and_cluster_filter_using_quads_R
# use this notebook to annotate all trio and sibling dnms with their mutation types 
# after this, run 19_distribution_genome_figure_files_R and 20_distribution_genome_figure_files_python
# code to generate and plot spectra are from natanael spisak

In [ ]:
import pandas as pd
import os
from google.cloud import storage
import re
import numpy as np 
import pandas as pd
pd.options.mode.chained_assignment = None
import os
import pyfastx
from liftover import get_lifter as Converter
import matplotlib.pyplot as plt


In [ ]:
# from natanael spisak's github repo: https://github.com/n-t-n-el/clock-like

class Reference():
    """ This class uses genomic reference """
    
    def __init__(self,ref=19,dirname=None):
        assert (ref==19) or (ref==38)
        if dirname is not None: self.dirname = dirname
        else: dirname = 'data/'
        
        self.hg = Fasta(dirname + 'hg{}.fa'.format(ref))
        self.hgRefChr = {}
        for i in list(range(1,22+1)) + ['X', 'Y']:
            key = 'chr{}'.format(i)
            self.hgRefChr.update({str(i): key})
    
    def findRef(self,Chr,Pos):
        return self.hg[self.hgRefChr[Chr]][Pos-1].seq.upper()

    def findContext(self,Chr,Pos):
        return self.hg[self.hgRefChr[Chr]][Pos-2:Pos+1].seq.upper()

    def check(self,df,chr_col='Chr',pos_col='Pos',pos_shift=0,ref_col='Ref'):
        df['Chr'] = df[chr_col].astype(str).str.replace('chr','')
        df['Pos'] = df[pos_col]+pos_shift
        df['Ref-check'] = df.apply(lambda x: self.findRef(x['Chr'], x['Pos']), axis=1)
        self.errors = df['Ref-check']!=df[ref_col]
        return sum(df['Ref-check']!=df[ref_col])/len(df)

conv = Converter('hg38', 'hg19')
def convertChr(Chr,Pos):
    l = conv.convert_coordinate(Chr,Pos)
    if len(l)>0: return l[0][0].replace('chr','')
    else: return np.nan 
def convertPos(Chr,Pos):
    l = conv.convert_coordinate(Chr,Pos)
    if len(l)>0: return int(l[0][1])
    else: return np.nan 
    
    
class COSMIC():
    """ This class introduces cosmic order and 
        cosmic-like plots for signatures """
    
    def __init__(self,loc='data/COSMIC_v3.3.1_SBS_GRCh38.txt'):
        
        self.order = ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T',
                      'C[C>A]A', 'C[C>A]C', 'C[C>A]G', 'C[C>A]T',
                      'G[C>A]A', 'G[C>A]C', 'G[C>A]G', 'G[C>A]T',
                      'T[C>A]A', 'T[C>A]C', 'T[C>A]G', 'T[C>A]T',
                      'A[C>G]A', 'A[C>G]C', 'A[C>G]G', 'A[C>G]T',
                      'C[C>G]A', 'C[C>G]C', 'C[C>G]G', 'C[C>G]T',
                      'G[C>G]A', 'G[C>G]C', 'G[C>G]G', 'G[C>G]T',
                      'T[C>G]A', 'T[C>G]C', 'T[C>G]G', 'T[C>G]T',
                      'A[C>T]A', 'A[C>T]C', 'A[C>T]G', 'A[C>T]T',
                      'C[C>T]A', 'C[C>T]C', 'C[C>T]G', 'C[C>T]T',
                      'G[C>T]A', 'G[C>T]C', 'G[C>T]G', 'G[C>T]T',
                      'T[C>T]A', 'T[C>T]C', 'T[C>T]G', 'T[C>T]T',
                      'A[T>A]A', 'A[T>A]C', 'A[T>A]G', 'A[T>A]T',
                      'C[T>A]A', 'C[T>A]C', 'C[T>A]G', 'C[T>A]T',
                      'G[T>A]A', 'G[T>A]C', 'G[T>A]G', 'G[T>A]T',
                      'T[T>A]A', 'T[T>A]C', 'T[T>A]G', 'T[T>A]T',
                      'A[T>C]A', 'A[T>C]C', 'A[T>C]G', 'A[T>C]T',
                      'C[T>C]A', 'C[T>C]C', 'C[T>C]G', 'C[T>C]T',
                      'G[T>C]A', 'G[T>C]C', 'G[T>C]G', 'G[T>C]T',
                      'T[T>C]A', 'T[T>C]C', 'T[T>C]G', 'T[T>C]T',
                      'A[T>G]A', 'A[T>G]C', 'A[T>G]G', 'A[T>G]T',
                      'C[T>G]A', 'C[T>G]C', 'C[T>G]G', 'C[T>G]T',
                      'G[T>G]A', 'G[T>G]C', 'G[T>G]G', 'G[T>G]T',
                      'T[T>G]A', 'T[T>G]C', 'T[T>G]G', 'T[T>G]T']
        self.contexts = [t[0]+t[2]+t[6] for t in self.order]
        self.colors = [[3/256,189/256,239/256],
                       [1/256,1/256,1/256],
                       [228/256,41/256,38/256],
                       [203/256,202/256,202/256],
                       [162/256,207/256,99/256],
                       [236/256,199/256,197/256]]
        
        bad_order = pd.read_table(loc)
        good_order = np.array(self.order).argsort().argsort()
        self.cosmic = pd.DataFrame(bad_order.to_numpy()[good_order], columns=bad_order.columns)
        assert (self.cosmic.Type==self.order).all()
        A,C,G,T = 'A','C','G','T'
        self.RC = {A: T, C: G, G: C, T: A}
        self.Alt = {A: [C,G,T], C: [A,G,T], G: [A,C,T], T: [A,C,G]}

    
    def revType(self,typ):
        return self.RC[typ[6]]+typ[1]+self.RC[typ[2]]+typ[3]+self.RC[typ[4]]+typ[5]+self.RC[typ[0]]
        
    def prepareFold(self,types_column):
        generalTypes = set(types_column)
        self.foldDict = dict(zip(generalTypes, generalTypes))
        for typ in generalTypes:
            if typ[2] in ['A','G']:
                self.foldDict[typ] = self.revType(typ)
        
    def fold(self,types_column):
        """ Map 192 to 96 mutation types
            A[C>T]G is A[C>T]G (COSMIC red)
            but so is  C[G>A]T """ 
        self.prepareFold(types_column)
        return types_column.map(self.foldDict)
        
    def plot(self, proba, save=None, title=None, show_letters=False, figsize=(6,2), width=1):
        """COSMIC-like plot"""
        fig, ax = plt.subplots(figsize=figsize)

        delta = len(self.order)//len(self.colors)
        x = np.arange(delta)

        for i in range(6):
            ax.bar(x+i*delta, proba[i*delta:(i+1)*delta],
                   color=self.colors[i], width=width)

        ax.set_xlim(-width/2, len(self.order)-width/2)
        ax.set_xticks(range(len(self.order)))

        if show_letters:
            ax.set_xticklabels(self.contexts, rotation=90)
        else:
            ax.set_xticklabels([''] * len(self.contexts), rotation=90)

        ax.set_xlabel('Mutation types')
        ax.set_ylabel('Distribution')
        ax.legend([], title=title, frameon=False, loc=1)
        ax.set_ylim(0, 8)
        plt.tight_layout()
        if save:
            plt.savefig(save)
        plt.show()


In [ ]:
cosmic = COSMIC()
fa = pyfastx.Fasta('hg38.fa.gz')
# read in full_qc_kept_mutations 
f_qc_aou = "./sibs_snps_QC_GIAB_distance_AF_outliers.txt"
qc_aou = pd.read_csv(f_qc_aou, sep = "\t")
qc_aou.loc[:, 'Context'] = ''
for i in range(qc_aou.shape[0]):
    chr_str = "chr" + str(qc_aou.loc[i,"chr"])
    Pos = qc_aou.loc[i,"pos"]
    qc_aou.loc[i, 'Context'] = fa[chr_str][Pos-2:Pos+1].seq.upper()
    

In [ ]:
f_qc_aou_trios = "./trios_snps_after_standard_QC_GIAB.txt"
qc_aou_trios = pd.read_csv(f_qc_aou_trios, sep = "\t")
qc_aou_trios.loc[:, 'Context'] = ''
for i in range(qc_aou_trios.shape[0]):
    chr_str = "chr" + str(qc_aou_trios.loc[i,"chr"])
    Pos = qc_aou_trios.loc[i,"pos"]
    qc_aou_trios.loc[i, 'Context'] = fa[chr_str][Pos-2:Pos+1].seq.upper()

In [ ]:
qc_aou['GeneralType'] = qc_aou['Context'].str[0]+'['+qc_aou['ref']+'>'+qc_aou['alt']+']'+qc_aou['Context'].str[2]
qc_aou['Type'] = cosmic.fold(qc_aou['GeneralType'])
qc_aou_trios['GeneralType'] = qc_aou_trios['Context'].str[0]+'['+qc_aou_trios['ref']+'>'+qc_aou_trios['alt']+']'+qc_aou_trios['Context'].str[2]
qc_aou_trios['Type'] = cosmic.fold(qc_aou_trios['GeneralType'])

In [ ]:
cosmic_to_plot_aou = pd.DataFrame(np.nan, index=range(len(cosmic.contexts)), columns=['context'])
cosmic_to_plot_trios = pd.DataFrame(np.nan, index=range(len(cosmic.contexts)), columns=['context'])
cosmic_to_plot_aou_all = pd.DataFrame(np.nan, index=range(len(cosmic.contexts)), columns=['context'])

In [ ]:
cosmic_to_plot_aou['context'] = cosmic.order
cosmic_to_plot_aou['percent'] = np.nan
cosmic_to_plot_trios['context'] = cosmic.order
cosmic_to_plot_trios['percent'] = np.nan
cosmic_to_plot_aou_all['context'] = cosmic.order
cosmic_to_plot_aou_all['percent'] = np.nan

In [ ]:
tot_aou = qc_aou.shape[0]
tot_trios = qc_aou_trios.shape[0]
tot_aou_all = tot_aou + tot_trios
for i in range(len(cosmic.order)):
    count_aou = qc_aou[qc_aou['Type'] == cosmic.order[i]].shape[0]
    cosmic_to_plot_aou.loc[i,'percent'] = count_aou/tot_aou*100
    count_trios = qc_aou_trios[qc_aou_trios['Type'] == cosmic.order[i]].shape[0]
    cosmic_to_plot_trios.loc[i,'percent'] = count_trios/tot_trios*100
    count_aou_all = count_aou + count_trios
    cosmic_to_plot_aou_all.loc[i,'percent'] = count_aou_all/tot_aou_all*100

In [ ]:
cosmic.plot(proba = cosmic_to_plot_aou['percent'], figsize=(12,4),
             width=0.7, show_letters = True, title="AoU Siblings")

cosmic.plot(proba = cosmic_to_plot_trios['percent'], figsize=(12,4),
             width=0.7, show_letters = True, title="AoU Trios")
cosmic.plot(proba = cosmic_to_plot_aou_all['percent'], figsize=(12,4),
             width=0.7, show_letters = True, title="AoU Siblings+Trios")


In [ ]:
qc_aou.to_csv("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt", sep="\t",
             index=False)
qc_aou_trios.to_csv("./trios_snps_after_standard_QC_GIAB_COSMIC.txt", sep="\t",
             index=False)